# Synapse Notebook: PySpark - Reading ADLS & Creating Tables
**Language:** Python (PySpark)  
**Attach to:** Spark pool (e.g., sparkpool1)  
**Purpose:** Read data from ADLS, transform, and create tables in Synapse

## CELL 1: Initialize and set up storage connection

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType
import datetime

spark = SparkSession.builder.appName("SynapseDemo").getOrCreate()

# Display Spark version
print(f"Spark Version: {spark.version}")

## CELL 2: Set up ADLS connection credentials

In [ ]:
# Method 1: Using storage account key
spark.conf.set(
    "fs.azure.account.key.storageaccount.dfs.core.windows.net",
    "your-access-key-or-sas-token"
)

# Method 2: Using managed identity (recommended in Synapse)
# spark.conf.set("fs.azure.auth.type", "OAuth")
# spark.conf.set("fs.azure.auth.oauth.provider.type", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

adls_path = "abfss://container-name@storageaccount.dfs.core.windows.net"
print(f"ADLS Path configured: {adls_path}")

## CELL 3: Read CSV file from ADLS

In [ ]:
df_sales_csv = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("delimiter", ",") \
    .load(f"{adls_path}/raw/sales/")

# Display the data
df_sales_csv.show(10)
df_sales_csv.printSchema()

## CELL 4: Read Parquet file from ADLS

In [ ]:
df_sales_parquet = spark.read \
    .format("parquet") \
    .load(f"{adls_path}/processed/sales_parquet/")

df_sales_parquet.show(5)
print(f"Parquet file row count: {df_sales_parquet.count()}")

## CELL 5: Read JSON file from ADLS

In [ ]:
df_sales_json = spark.read \
    .format("json") \
    .option("multiline", True) \
    .load(f"{adls_path}/raw/sales_json/")

df_sales_json.show()
df_sales_json.printSchema()

## CELL 6: Create DataFrame with explicit schema

In [ ]:
schema = StructType([
    StructField("SalesID", IntegerType(), False),
    StructField("CustomerID", IntegerType(), False),
    StructField("ProductName", StringType(), True),
    StructField("SalesAmount", DecimalType(10, 2), True),
    StructField("SalesDate", DateType(), True)
])

df_sales_explicit = spark.read \
    .format("csv") \
    .schema(schema) \
    .option("header", True) \
    .load(f"{adls_path}/raw/sales/")

df_sales_explicit.show()
df_sales_explicit.printSchema()

## CELL 7: Data transformation and cleanup

In [ ]:
from pyspark.sql.functions import col, trim, upper, to_date, year, month

df_cleaned = df_sales_csv \
    .withColumn("ProductName", trim(upper(col("ProductName")))) \
    .withColumn("SalesDate", to_date(col("SalesDate"), "yyyy-MM-dd")) \
    .withColumn("Year", year(col("SalesDate"))) \
    .withColumn("Month", month(col("SalesDate"))) \
    .filter(col("SalesAmount") > 0)

df_cleaned.show()
print(f"Cleaned data: {df_cleaned.count()} rows")

## CELL 8: Create temporary view (session-scoped)

In [ ]:
df_cleaned.createOrReplaceTempView("vw_sales_temp")

# Query using SQL
result = spark.sql("SELECT * FROM vw_sales_temp WHERE Year = 2024 LIMIT 10")
result.show()

## CELL 9: Create external table (permanent, stored in Synapse)

In [ ]:
df_cleaned.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", f"{adls_path}/processed/sales_external/") \
    .saveAsTable("sales_external")

print("External table 'sales_external' created")

## CELL 10: Create managed table (data stored in Synapse warehouse)

In [ ]:
df_cleaned.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sales_managed")

print("Managed table 'sales_managed' created")

## CELL 11: Register table and query it

In [ ]:
result = spark.sql("SELECT COUNT(*) as TotalSales FROM sales_managed")
result.show()

# Verify table exists
spark.sql("SHOW TABLES").filter("tableName LIKE '%sales%'").show()

## CELL 12: Aggregation operations

In [ ]:
df_aggregated = df_cleaned.groupBy("Year", "Month") \
    .agg({
        "SalesAmount": "sum",
        "CustomerID": "count",
        "ProductName": "countDistinct"
    }) \
    .withColumnRenamed("sum(SalesAmount)", "TotalRevenue") \
    .withColumnRenamed("count(CustomerID)", "TotalTransactions") \
    .withColumnRenamed("count_distinct(ProductName)", "UniqueProducts") \
    .orderBy("Year", "Month")

df_aggregated.show()

## CELL 13: Window functions

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead, sum as spark_sum

window_spec = Window.partitionBy("CustomerID").orderBy(col("SalesDate"))

df_with_window = df_cleaned.withColumn(
    "RunningTotal",
    spark_sum(col("SalesAmount")).over(window_spec)
).withColumn(
    "PrevSalesAmount",
    lag(col("SalesAmount")).over(window_spec)
).withColumn(
    "NextSalesDate",
    lead(col("SalesDate")).over(window_spec)
)

df_with_window.select("SalesID", "CustomerID", "SalesAmount", "RunningTotal", "PrevSalesAmount").show()

## CELL 14: Write to ADLS in different formats

In [ ]:
# Write as Parquet (columnar, compressed)
df_cleaned.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("Year") \
    .save(f"{adls_path}/processed/sales_parquet_partitioned/")

print("Parquet written with Year partition")

# Write as Delta Lake (with ACID transactions)
df_cleaned.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("Year") \
    .save(f"{adls_path}/processed/sales_delta/")

print("Delta Lake written with Year partition")

## CELL 15: Repartition data for optimization

In [ ]:
df_repartitioned = df_cleaned.repartition(4, col("Year"))
df_repartitioned.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(f"{adls_path}/processed/sales_repartitioned/")

print(f"Data repartitioned into 4 partitions and written to ADLS")
print(f"Number of output files: {df_repartitioned.rdd.getNumPartitions()}")

## CELL 16: Read back and verify

In [ ]:
df_verify = spark.read \
    .format("parquet") \
    .load(f"{adls_path}/processed/sales_parquet_partitioned/")

print(f"Total rows in verified data: {df_verify.count()}")
df_verify.show(5)
df_verify.printSchema()

## BONUS: Performance tips and best practices

In [ ]:
# Enable caching for frequently used DataFrames
df_cleaned.cache()
print(f"DataFrame cached. Memory usage info:")
df_cleaned.storageLevel

# Check query plan
df_cleaned.filter(col("SalesAmount") > 1000).explain(extended=True)

# Unpersist when done
# df_cleaned.unpersist()